# 01 - VLM to VLA Training Loop\n\nThis notebook makes the architectural bridge concrete without requiring a heavy model setup.

## Concept\n\nVLM training example:\n\n`image + question -> answer tokens`\n\nVLA training example:\n\n`camera views + instruction + robot state -> action chunk`

In [ ]:
import torch\nimport torch.nn as nn\n\nbatch = 4\nimage_embedding_dim = 768\ntext_embedding_dim = 512\nstate_dim = 16\naction_dim = 7\nchunk_len = 8\n\nvision_tokens = torch.randn(batch, image_embedding_dim)\ninstruction_tokens = torch.randn(batch, text_embedding_dim)\nrobot_state = torch.randn(batch, state_dim)\ntarget_action_chunk = torch.randn(batch, chunk_len, action_dim)\n\nclass TinyVLAHead(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Linear(image_embedding_dim + text_embedding_dim + state_dim, 256),\n            nn.ReLU(),\n            nn.Linear(256, chunk_len * action_dim),\n        )\n\n    def forward(self, vision, instruction, state):\n        x = torch.cat([vision, instruction, state], dim=-1)\n        return self.net(x).view(-1, chunk_len, action_dim)\n\nmodel = TinyVLAHead()\npred = model(vision_tokens, instruction_tokens, robot_state)\nloss = nn.MSELoss()(pred, target_action_chunk)\n\nprint('predicted action chunk:', tuple(pred.shape))\nprint('behavior cloning loss:', float(loss))

## What to say\n\n> For VLAs, the model architecture is only part of the story. The training rows now include observations, language, robot state, and actions. That makes dataset quality, embodiment, latency, and safety central design choices.